# Question 1: Search Algorithms — The Warehouse Logistics Bot

**Course:** Artificial Intelligence (ARI711S)  
**Assignment:** 1  
**Group:** Ninjas 2026

---

## Overview

In this task, we implement an **Automated Guided Vehicle (AGV)** navigation system for a warehouse robot. The robot must find the most efficient path from a **Charging Station (A)** to a **Product Bin (B)**.

We implement and compare two **informed search algorithms**:
- **Greedy Best-First Search** — uses only the heuristic `h(n)` (Euclidean distance to goal)
- **A\* Search** — uses `f(n) = g(n) + h(n)` (path cost + heuristic)

### Heuristic Function
We use **Euclidean Distance** as the heuristic:
$$h(n) = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}$$

where $(x_1, y_1)$ is the current node and $(x_2, y_2)$ is the goal.

---

## 1. Imports and Setup

We import standard Python libraries. `heapq` provides the **priority queue** (min-heap) needed for both Greedy and A* frontiers. `matplotlib` is used for visual output.

In [ ]:
import heapq
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

---
## 2. Node Class

The `Node` class stores:
- `state` — the grid position `(row, col)` of the robot
- `parent` — the node that generated this one (used to reconstruct the path)
- `action` — the direction taken to reach this node (`up`, `down`, `left`, `right`)
- `g` — the **path cost** from the start to this node (number of steps)

We also define comparison operators so nodes can be ordered in the priority queue.

In [ ]:
class Node:
    """
    Represents a single state in the search tree.

    Attributes:
        state  (tuple): Grid coordinates (row, col).
        parent (Node) : The node from which this node was reached.
        action (str)  : The movement taken to reach this node.
        g      (int)  : Path cost from the start node to this node.
    """

    def __init__(self, state, parent=None, action=None, g=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.g = g  # path cost

    def __lt__(self, other):
        """Allow comparison for heapq ordering."""
        return self.g < other.g

    def __eq__(self, other):
        return isinstance(other, Node) and self.state == other.state

    def __hash__(self):
        return hash(self.state)

    def path(self):
        """Reconstruct the path from start to this node by tracing parents."""
        nodes = []
        current = self
        while current is not None:
            nodes.append(current)
            current = current.parent
        return list(reversed(nodes))

---
## 3. Warehouse Class

The `Warehouse` class is responsible for:
1. **Parsing** the `.txt` layout file into a grid
2. **Locating** the start (`A`) and goal (`B`) positions
3. **Storing** wall/shelving positions
4. **Providing** valid neighbouring cells for movement
5. **Running** Greedy or A* search via the `solve()` method

In [ ]:
class Warehouse:
    """
    Models the warehouse environment.

    The layout is read from a .txt file where:
        '#' = shelving unit (wall)
        'A' = charging station (start)
        'B' = product bin (goal)
        ' ' = open aisle
    """

    def __init__(self, filename):
        """Read and parse the warehouse layout from a text file."""
        self.walls = set()
        self.start = None
        self.goal = None
        self.grid = []

        with open(filename, 'r') as f:
            lines = f.read().splitlines()

        self.rows = len(lines)
        self.cols = max(len(line) for line in lines)

        for row, line in enumerate(lines):
            row_data = []
            for col, char in enumerate(line):
                row_data.append(char)
                if char == '#':
                    self.walls.add((row, col))
                elif char == 'A':
                    self.start = (row, col)
                elif char == 'B':
                    self.goal = (row, col)
            # Pad shorter rows with spaces
            while len(row_data) < self.cols:
                row_data.append(' ')
            self.grid.append(row_data)

        if self.start is None:
            raise ValueError("No start position 'A' found in the warehouse layout.")
        if self.goal is None:
            raise ValueError("No goal position 'B' found in the warehouse layout.")

        print(f"Warehouse loaded: {self.rows} rows x {self.cols} cols")
        print(f"Charging Station (A): {self.start}")
        print(f"Product Bin     (B): {self.goal}")
        print(f"Shelving units (#):  {len(self.walls)} cells")

    # ------------------------------------------------------------------
    # Heuristic: Euclidean Distance
    # ------------------------------------------------------------------
    def heuristic(self, state):
        """
        Euclidean distance from 'state' to the goal.

            h(n) = sqrt((x1 - x2)^2 + (y1 - y2)^2)
        """
        r1, c1 = state
        r2, c2 = self.goal
        return math.sqrt((r1 - r2) ** 2 + (c1 - c2) ** 2)

    # ------------------------------------------------------------------
    # Neighbours
    # ------------------------------------------------------------------
    def neighbors(self, state):
        """
        Return valid adjacent cells (up, down, left, right).
        A cell is valid if it is within bounds and is not a shelving unit.
        """
        row, col = state
        directions = [
            ('up',    row - 1, col),
            ('down',  row + 1, col),
            ('left',  row,     col - 1),
            ('right', row,     col + 1),
        ]
        valid = []
        for action, r, c in directions:
            if 0 <= r < self.rows and 0 <= c < self.cols:
                if (r, c) not in self.walls:
                    valid.append((action, (r, c)))
        return valid

    # ------------------------------------------------------------------
    # Solve
    # ------------------------------------------------------------------
    def solve(self, algorithm="astar"):
        """
        Search for a path from A to B using the specified algorithm.

        Parameters:
            algorithm (str): 'greedy' for Greedy Best-First Search,
                             'astar'  for A* Search.

        Returns:
            path     (list): Ordered list of Node objects from A to B.
            explored (set) : All states that were expanded during search.

        Raises:
            ValueError if no path exists.
        """
        algorithm = algorithm.lower()
        if algorithm not in ("greedy", "astar"):
            raise ValueError("Algorithm must be 'greedy' or 'astar'.")

        start_node = Node(state=self.start, g=0)

        # Priority queue entries: (priority, tie_breaker, node)
        # tie_breaker is a counter to ensure stable ordering when priorities tie
        counter = 0
        frontier = []

        if algorithm == "greedy":
            priority = self.heuristic(self.start)   # f(n) = h(n)
        else:
            priority = 0 + self.heuristic(self.start)  # f(n) = g(n) + h(n)

        heapq.heappush(frontier, (priority, counter, start_node))

        # Track best-known g(n) for each state to avoid re-expanding
        best_g = {self.start: 0}
        explored = set()

        while frontier:
            _, _, current = heapq.heappop(frontier)

            # Goal test
            if current.state == self.goal:
                return current.path(), explored

            # Skip if already expanded with a better cost
            if current.state in explored:
                continue
            explored.add(current.state)

            for action, neighbor_state in self.neighbors(current.state):
                new_g = current.g + 1  # uniform step cost

                # Only add to frontier if we found a better path to this state
                if neighbor_state not in best_g or new_g < best_g[neighbor_state]:
                    best_g[neighbor_state] = new_g
                    child = Node(
                        state=neighbor_state,
                        parent=current,
                        action=action,
                        g=new_g
                    )
                    h = self.heuristic(neighbor_state)
                    if algorithm == "greedy":
                        priority = h           # Greedy: only heuristic
                    else:
                        priority = new_g + h   # A*: path cost + heuristic

                    counter += 1
                    heapq.heappush(frontier, (priority, counter, child))

        raise ValueError("No path found from A to B.")

    # ------------------------------------------------------------------
    # Visualise
    # ------------------------------------------------------------------
    def visualise(self, path, explored, algorithm_name, save_as="warehouse_path.png"):
        """
        Render the warehouse grid and save the image.

        Colour legend:
            Dark grey  — Shelving unit (wall)
            Light grey — Explored aisle (visited but not on final path)
            Cyan       — Optimal path
            Green      — Charging Station (A)
            Red        — Product Bin (B)
            White      — Unexplored open aisle
        """
        # Build colour grid
        COLOR_OPEN     = [1.00, 1.00, 1.00]  # white
        COLOR_WALL     = [0.20, 0.20, 0.20]  # dark grey
        COLOR_EXPLORED = [0.75, 0.90, 1.00]  # light blue
        COLOR_PATH     = [0.00, 0.85, 0.85]  # cyan
        COLOR_START    = [0.00, 0.80, 0.20]  # green
        COLOR_GOAL     = [0.90, 0.15, 0.15]  # red

        image = np.ones((self.rows, self.cols, 3))

        # Fill cells
        for r in range(self.rows):
            for c in range(self.cols):
                if (r, c) in self.walls:
                    image[r, c] = COLOR_WALL
                else:
                    image[r, c] = COLOR_OPEN

        # Mark explored states
        for state in explored:
            r, c = state
            if state != self.start and state != self.goal:
                image[r, c] = COLOR_EXPLORED

        # Mark path
        path_states = {node.state for node in path}
        for node in path:
            r, c = node.state
            if node.state != self.start and node.state != self.goal:
                image[r, c] = COLOR_PATH

        # Mark start and goal
        image[self.start[0], self.start[1]] = COLOR_START
        image[self.goal[0],  self.goal[1]]  = COLOR_GOAL

        # Plot
        fig, ax = plt.subplots(figsize=(max(8, self.cols // 2), max(6, self.rows // 2)))
        ax.imshow(image, interpolation='nearest')

        # Annotate start and goal
        ax.text(self.start[1], self.start[0], 'A',
                ha='center', va='center', fontsize=9, fontweight='bold', color='white')
        ax.text(self.goal[1], self.goal[0], 'B',
                ha='center', va='center', fontsize=9, fontweight='bold', color='white')

        # Grid lines
        ax.set_xticks(np.arange(-0.5, self.cols, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, self.rows, 1), minor=True)
        ax.grid(which='minor', color='#cccccc', linewidth=0.4)
        ax.tick_params(which='both', bottom=False, left=False,
                       labelbottom=False, labelleft=False)

        # Legend
        legend_items = [
            mpatches.Patch(color=[c/1 for c in COLOR_START],    label='Charging Station (A)'),
            mpatches.Patch(color=[c/1 for c in COLOR_GOAL],     label='Product Bin (B)'),
            mpatches.Patch(color=[c/1 for c in COLOR_PATH],     label='Optimal Path'),
            mpatches.Patch(color=[c/1 for c in COLOR_EXPLORED], label='Explored (not on path)'),
            mpatches.Patch(color=[c/1 for c in COLOR_WALL],     label='Shelving Unit (Wall)'),
        ]
        ax.legend(handles=legend_items, loc='upper right',
                  fontsize=8, framealpha=0.9, bbox_to_anchor=(1.35, 1.0))

        ax.set_title(f"Warehouse Robot — {algorithm_name}\n"
                     f"Path length: {len(path) - 1} steps  |  "
                     f"States explored: {len(explored)}",
                     fontsize=11)

        plt.tight_layout()
        plt.savefig(save_as, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Image saved as '{save_as}'")

---
## 4. Load the Warehouse

We load the warehouse from `warehouse.txt`. The file uses:
- `#` — shelving unit (impassable)
- `A` — charging station (start)
- `B` — product bin (goal)
- ` ` — open aisle (passable)

In [ ]:
# Load the warehouse layout
# If you downloaded a different warehouse.txt from the course portal,
# replace 'warehouse.txt' with its filename.
warehouse = Warehouse('warehouse.txt')

---
## 5. Greedy Best-First Search

**How it works:**  
The frontier is a priority queue ordered by the heuristic value `h(n)` alone (Euclidean distance to goal). The algorithm always expands the node that *looks* closest to the goal.

$$\text{Priority} = h(n) = \sqrt{(row_1 - row_2)^2 + (col_1 - col_2)^2}$$

**Limitation:** Greedy search is not guaranteed to find the *optimal* (shortest) path — it can be misled by the heuristic if the direct route is blocked.

In [ ]:
print("=" * 50)
print("GREEDY BEST-FIRST SEARCH")
print("=" * 50)

greedy_path, greedy_explored = warehouse.solve(algorithm="greedy")

print(f"\nPath found!")
print(f"  Steps in path : {len(greedy_path) - 1}")
print(f"  States explored: {len(greedy_explored)}")
print(f"\nPath coordinates:")
for node in greedy_path:
    print(f"  {node.state}  (action: {node.action or 'START'}, g={node.g})")

In [ ]:
# Visualise Greedy result
warehouse.visualise(
    path=greedy_path,
    explored=greedy_explored,
    algorithm_name="Greedy Best-First Search",
    save_as="warehouse_path_greedy.png"
)

---
## 6. A* Search

**How it works:**  
A* uses the evaluation function:

$$f(n) = g(n) + h(n)$$

where:
- `g(n)` = actual path cost from start to node `n`
- `h(n)` = Euclidean distance heuristic from `n` to goal

By combining real cost with the heuristic estimate, A* is **both complete and optimal** (guaranteed to find the shortest path), provided the heuristic is *admissible* (never overestimates the true cost). Euclidean distance in a grid where each move costs 1 is admissible since straight-line distance ≤ actual path distance.

In [ ]:
print("=" * 50)
print("A* SEARCH")
print("=" * 50)

astar_path, astar_explored = warehouse.solve(algorithm="astar")

print(f"\nPath found!")
print(f"  Steps in path : {len(astar_path) - 1}")
print(f"  States explored: {len(astar_explored)}")
print(f"\nPath coordinates:")
for node in astar_path:
    print(f"  {node.state}  (action: {node.action or 'START'}, g={node.g})")

In [ ]:
# Visualise A* result — also saved as the required 'warehouse_path.png'
warehouse.visualise(
    path=astar_path,
    explored=astar_explored,
    algorithm_name="A* Search",
    save_as="warehouse_path.png"
)

---
## 7. Comparison: Greedy vs A*

Below we compare both algorithms side-by-side in a single figure.

In [ ]:
def build_image(warehouse, path, explored):
    """Helper: build a NumPy colour array for a given path/explored result."""
    COLOR_OPEN     = [1.00, 1.00, 1.00]
    COLOR_WALL     = [0.20, 0.20, 0.20]
    COLOR_EXPLORED = [0.75, 0.90, 1.00]
    COLOR_PATH     = [0.00, 0.85, 0.85]
    COLOR_START    = [0.00, 0.80, 0.20]
    COLOR_GOAL     = [0.90, 0.15, 0.15]

    image = np.ones((warehouse.rows, warehouse.cols, 3))
    for r in range(warehouse.rows):
        for c in range(warehouse.cols):
            image[r, c] = COLOR_WALL if (r, c) in warehouse.walls else COLOR_OPEN
    for state in explored:
        if state not in (warehouse.start, warehouse.goal):
            image[state[0], state[1]] = COLOR_EXPLORED
    for node in path:
        if node.state not in (warehouse.start, warehouse.goal):
            image[node.state[0], node.state[1]] = COLOR_PATH
    image[warehouse.start[0], warehouse.start[1]] = COLOR_START
    image[warehouse.goal[0],  warehouse.goal[1]]  = COLOR_GOAL
    return image


fig, axes = plt.subplots(1, 2, figsize=(18, 8))

results = [
    ("Greedy Best-First Search", greedy_path, greedy_explored),
    ("A* Search",                astar_path,  astar_explored),
]

for ax, (name, path, explored) in zip(axes, results):
    img = build_image(warehouse, path, explored)
    ax.imshow(img, interpolation='nearest')
    ax.set_title(
        f"{name}\n"
        f"Path: {len(path)-1} steps  |  Explored: {len(explored)} states",
        fontsize=11
    )
    ax.text(warehouse.start[1], warehouse.start[0], 'A',
            ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    ax.text(warehouse.goal[1], warehouse.goal[0], 'B',
            ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    ax.set_xticks([])
    ax.set_yticks([])

# Shared legend
legend_items = [
    mpatches.Patch(color=[0.00, 0.80, 0.20], label='Charging Station (A)'),
    mpatches.Patch(color=[0.90, 0.15, 0.15], label='Product Bin (B)'),
    mpatches.Patch(color=[0.00, 0.85, 0.85], label='Optimal Path'),
    mpatches.Patch(color=[0.75, 0.90, 1.00], label='Explored (not on path)'),
    mpatches.Patch(color=[0.20, 0.20, 0.20], label='Shelving Unit (Wall)'),
]
fig.legend(handles=legend_items, loc='lower center',
           ncol=5, fontsize=9, framealpha=0.9)
plt.suptitle("Warehouse Robot Navigation — Algorithm Comparison", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("warehouse_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("Comparison image saved as 'warehouse_comparison.png'")

---
## 8. Summary and Analysis

| Metric | Greedy Best-First | A* |
|--------|------------------|----|
| **Priority function** | `h(n)` only | `g(n) + h(n)` |
| **Optimality** | Not guaranteed | Guaranteed (admissible heuristic) |
| **Completeness** | Not complete (can loop) | Complete |
| **Heuristic used** | Euclidean distance | Euclidean distance |

### Key Observations

- **Greedy Best-First Search** is fast and explores fewer nodes when the heuristic guides well, but it may find a suboptimal path because it ignores the actual path cost `g(n)`. It behaves like a "rush towards the goal" strategy.

- **A\* Search** explores more states in some cases, but guarantees the **shortest possible path**. The Euclidean distance heuristic is **admissible** (never overestimates) because a straight-line distance is always ≤ the grid path distance, ensuring optimality.

- In warehouse logistics, A* is the preferred algorithm because **path optimality directly reduces fuel/energy consumption** and time, which is critical for AGV operations at scale.

### Heuristic Admissibility
The Euclidean distance $h(n) = \sqrt{(r_1-r_2)^2 + (c_1-c_2)^2}$ is admissible because:
- In a grid with unit step costs, the minimum possible steps between two cells ≥ their Euclidean distance
- Therefore $h(n) \leq h^*(n)$ (true cost), satisfying the admissibility condition for A* optimality